# Ring phenotype detector: P0 / P1 v0.2

CPU can run the default pretrained yeast brightfield segmentation; GPU is optional. No training/fine-tuning. Proposals require human review. This notebook runs exactly the repository CLI; raw data remain in your runtime/Drive.


In [ ]:
!git clone https://github.com/wangherm/ring-phenotype-detector.git
%cd /content/ring-phenotype-detector
%pip install -r requirements-cellpose.txt
!python -m pytest -q


## Choose the data path
Upload TEST.zip below, or mount Google Drive and set INPUT to its path. Only run the upload cell if needed.


In [ ]:
from google.colab import files
files.upload()  # choose TEST.zip
INPUT = '/content/ring-phenotype-detector/TEST.zip'
OUT = '/content/ring-results'


In [ ]:
from pathlib import Path
import subprocess, sys
assert Path(INPUT).exists(), INPUT
def run(*args):
    subprocess.run([sys.executable, '-m', 'ring_detector.cli', *args], check=True)
run('audit', '--input', INPUT, '--output', OUT+'/audit')
run('segment', '--manifest', OUT+'/audit/manifest.csv', '--config', 'configs/default.yaml', '--output', OUT+'/segments')
run('predict', '--baseline-only', '--manifest', OUT+'/audit/manifest.csv', '--segments', OUT+'/segments', '--config', 'configs/default.yaml', '--output', OUT+'/proposals')
run('review-export', '--proposals', OUT+'/proposals', '--output', OUT+'/review')


In [ ]:
from IPython.display import display, Image
for p in sorted(Path(OUT, 'proposals', 'overlays').glob('*.png'))[:6]:
    print(p.name)
    display(Image(filename=str(p), width=650))


## Download review package
Unzip, open review.html locally, view separate brightfield/fluorescence panels, label positive / negative / uncertain / invalid and download annotations.csv. Old masks/labels must not be reused by cell_id alone; v0.2 uses cell_uid. Inspect yellow/unassigned ring candidates separately in unassigned_rings.csv. Columns retain FOV/cell identities. Do not train on uncertain/invalid. Full masks and raw crops remain in OUT; save to Drive if needed.


In [ ]:
import shutil
review_zip = shutil.make_archive('/content/ring-review', 'zip', OUT+'/review')
files.download(review_zip)


## After review
Return annotations.csv to Codex before P2. Future training splits must group all cells from the same FOV and prefer date/session holdout. Color permutation tests are software checks, not independent dye generalization evidence.
